# Customer Churn Analysis — Task 1

**Decision question:** Which customer groups should a fictional subscription team contact first to reduce avoidable churn?

This notebook profiles, cleans, validates, and explores the supplied synthetic customer churn dataset. Findings are descriptive only; they do not establish causation or predict future churn.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
df = pd.read_csv('customer_churn_sample.csv')
df.head()

## 1. Data profile
Check shape, columns, data types, missing values, and duplicates.

In [ ]:
print('Shape:', df.shape)
print('\nData types:')
print(df.dtypes)
print('\nMissing values:')
print(df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())
print('Duplicate CustomerID values:', df['CustomerID'].duplicated().sum())

## 2. Quality decisions and cleaning log
- The supplied file has 15 rows and 11 columns.
- No missing cells were found, so no missing-value imputation was needed.
- No duplicate rows or duplicate CustomerID values were found, so no rows were deleted.
- Text fields were trimmed and Churn labels were standardized to Yes/No.
- Numeric fields were checked for impossible values.
- IQR was used to screen numeric columns for extreme values. No IQR outliers were found.
- TotalCharges was checked against MonthlyCharges × TenureMonths; all 15 rows matched within ₹0.01.
- No inconvenient rows were silently removed. The clean dataset therefore retains all 15 records.

In [ ]:
clean = df.copy()
for c in ['CustomerID','Gender','SubscriptionType','ContractType','PaymentMethod','Churn']:
    clean[c] = clean[c].astype(str).str.strip()
clean['Gender'] = clean['Gender'].str.title()
clean['Churn'] = clean['Churn'].str.title()
print('Missing cells:', clean.isna().sum().sum())
print('Duplicate rows:', clean.duplicated().sum())
print('Duplicate CustomerID:', clean['CustomerID'].duplicated().sum())
print('Invalid age:', ((clean['Age'] < 18) | (clean['Age'] > 100)).sum())
print('Invalid tenure:', (clean['TenureMonths'] < 0).sum())
print('Invalid monthly charge:', (clean['MonthlyCharges'] < 0).sum())
print('Invalid support tickets:', (clean['SupportTickets'] < 0).sum())
print('Total charge mismatches:', (~np.isclose(clean['TotalCharges'], clean['MonthlyCharges']*clean['TenureMonths'], atol=0.01)).sum())

## 3. Summary statistics

In [ ]:
clean[['Age','TenureMonths','MonthlyCharges','TotalCharges','SupportTickets']].describe().round(2)

## 4. Initial patterns
### Overall churn
7 of 15 customers churned (46.7%).

### Contract type
All 7 month-to-month customers churned in this small sample, while none of the 4 one-year or 4 two-year customers churned.

### Subscription type
Basic had 5 churned out of 7 customers (71.4%), Pro had 2 of 4 (50%), and Enterprise had 0 of 4.

### Support tickets
Churned customers averaged 4.29 support tickets versus 1.00 for non-churned customers.

### Tenure
Churned customers averaged 7.0 months of tenure versus 29.13 months for non-churned customers.

These are descriptive associations in only 15 synthetic records. They should not be interpreted as causal effects or a predictive model.

In [ ]:
for c in ['SubscriptionType','ContractType','PaymentMethod']:
    summary = clean.groupby(c).agg(Customers=('CustomerID','count'), Churned=('Churn', lambda x:(x=='Yes').sum()))
    summary['Churn_Rate'] = (summary['Churned']/summary['Customers']).round(3)
    display(summary)

print('Average support tickets by churn:')
display(clean.groupby('Churn')['SupportTickets'].mean().round(2))

print('Average tenure by churn:')
display(clean.groupby('Churn')['TenureMonths'].mean().round(2))

In [ ]:
clean['Churn'].value_counts().reindex(['No','Yes']).plot(kind='bar', title='Customer Churn Distribution')
plt.xlabel('Churn'); plt.ylabel('Customers'); plt.tight_layout(); plt.show()

clean.groupby('ContractType')['Churn'].apply(lambda x:(x=='Yes').mean()).reindex(['Month-to-Month','One Year','Two Year']).plot(kind='bar', title='Churn Rate by Contract Type')
plt.xlabel('Contract Type'); plt.ylabel('Churn Rate'); plt.ylim(0,1.05); plt.tight_layout(); plt.show()

clean.groupby('Churn')['SupportTickets'].mean().reindex(['No','Yes']).plot(kind='bar', title='Average Support Tickets by Churn Status')
plt.xlabel('Churn'); plt.ylabel('Average Tickets'); plt.tight_layout(); plt.show()

## 5. Final recommendation
**Audience:** The fictional subscription retention/support team.

**Decision:** Contact **month-to-month customers first**, especially those with **short tenure and higher recent support-ticket counts**.

**Supporting metrics:** In this sample, month-to-month churn was 7/7 (100%); churned customers averaged 4.29 support tickets and 7.0 months tenure.

**Uncertainty:** The dataset contains only 15 synthetic customers, so the rates are unstable and may not generalize. Contract type, support tickets, and tenure may also be related to one another; this analysis does not establish causation.

**Next measurement:** On a larger real dataset, track monthly churn rate by contract type and monitor whether targeted retention contacts reduce churn among the contacted group.